# Reproducing Sabatini & Kaufman (2024) on DANDI:000070

**Paper.** Sabatini, D. A. & Kaufman, M. T. (2024). Reach-dependent reorientation of rotational dynamics
in motor cortex. *Nature Communications* 15, 7007. doi:10.1038/s41467-024-51308-7. The claims file
(`claims/sabatini-2024.yaml`) was written from the version of record (PMC11325044) with its Methods and
its supplement, and from the paper's code (github.com/kaufmanlab/LDR-public at commit b65f28d).

**Dataset.** DANDI:000070 ("Neural population dynamics during reaching"), version **0.260528.0919**,
streamed. The paper cites the dandiset without a version. Of its three versions (0.251218.1714,
0.260528.0919 and the draft), the two published ones list the same ten assets with the same asset ids,
sizes and modification times, so they hold the same files; the most recent published one is pinned, as
the protocol asks when no version is cited. The two files the paper's Data availability statement
names are read: `sub-Jenkins_ses-20090918` (monkey J) and `sub-Nitschke_ses-20100923` (monkey N).

**What is tested.** The four datasets of the paper (M1-J, PMd-J, M1-N, PMd-N) are rebuilt from the
units, trials and hand position in the two files. The dataset holds no EMG.

| Claim | Tested here |
|---|---|
| data-composition | tested |
| jpca-fit-r2, jpca-variance-explained, single-lds-variance | tested |
| fig2c-condition-lds-variance, fig3a-eigenvalue-clusters, fig3c-eigenvalue-distance-auc, suppfig2-eigenvalues-from-kinematics | tested |
| fig3g-basis-frequencies, tbf-variance-explained, tbf-shuffle-control, fig3h-recoverability, fig3i-duration-warping, suppfig3-model-comparison | tested |
| fig4b-plane-alignment, fig4c-subspace-excursion-angles, rotation-orientation-dimensionality, fig4d-jpca-variance-correlation, fig4f-plane-alignment-explains-jpca | tested |
| fig5a-frequencies-segregated, fig5c-rotations-predict-rotations | tested |
| suppfig6a-location-alignment, suppfig6b-location-dimensionality, suppfig6c-location-distinct-from-rotations, fig6b-location-predicts-rotations, fig6c-location-tracks-direction, fig6e-kinematics-predict-location | tested |
| fig7c-ldr-encoding, suppfig7a-not-memorization, fig7c-standard-encoding-worse | tested |
| fig7f-ldr-decoding, trial-curvature-decoding, suppfig8-no-output-potent-dependence | tested (suppfig8 without EMG) |
| fig7f-decoder-beats-standard | tested except the GPFA decoders |

Where a parameter comes from (the claim, a passage of the paper, an assumption, or a standard choice)
is said in the docstrings of the helper `ldr.py` and beside the code here.

In [ ]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[_v] = "1"  # many small fits, some in parallel worker processes (protocol)
import time
import zlib
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import ldr

T_START = time.time()
DATASETS = [("J", "M1"), ("J", "PMd"), ("N", "M1"), ("N", "PMd")]
NAME = {d: f"{d[1]}-{d[0]}" for d in DATASETS}


def seed(claim):
    """The claim's seed (protocol): crc32 of '<paper key>/<claim id>'."""
    return zlib.crc32(f"sabatini-2024/{claim}".encode())


def seeds5(claim):
    return [seed(claim) + i for i in (0, 1000, 2000, 3000, 4000)]


print("dandiset", ldr.DANDISET, "version", ldr.VERSION)

## Loading

Each session is streamed once. Spike times are binned at 1 ms from 500 ms before to 1000 ms after
movement onset, smoothed with a Gaussian of 20 ms s.d. and sampled every 10 ms from -350 to +850 ms
(Methods, Subjects, behavior and data pre-processing; claims, shared method). The time-shuffled
copies for tbf-shuffle-control are made at load time under that claim's five seeds, and the warped
copies for fig3i at the same time.

In [ ]:
SESS = {}
for m in ("J", "N"):
    t0 = time.time()
    SESS[m] = ldr.load_session(m, shuffle_seeds=seeds5("tbf-shuffle-control"))
    print(m, "loaded in", round(time.time() - t0), "s; trials kept (successful, 72 conditions):",
          len(SESS[m]["trials"]["maze_condition"]))

## Data check

In [ ]:
import resource
for m in ("J", "N"):
    S = SESS[m]
    d = S["all_trials"]
    tr = S["trials"]
    print(f"== monkey {m} (asset {S['info']['asset_id']})")
    print("  trials in file", S["info"]["n_all_trials"], "| successful", int(d["task_success"].sum()),
          "| maze_condition values", len(np.unique(d["maze_condition"])),
          "(min", d["maze_condition"].min(), "max", d["maze_condition"].max(), ")")
    print("  units", len(S["area"]), {a: int((S["area"] == a).sum()) for a in ("M1", "PMd")})
    sp = S["info"]["spikes_per_unit"]
    print("  spikes per unit: min", sp.min(), "median", int(np.median(sp)), "max", sp.max())
    print("  recording segments (backward jumps of hand timestamps + 1):", S["info"]["n_hand_segments"],
          "| bounds (s):", [(round(a, 1), round(b, 1)) for a, b in S["info"]["hand_segment_bounds"]])
    print("  backward jumps in spike times per unit:", np.bincount(S["info"]["unit_jumps"]))
    print("  trials per segment:", np.bincount(d["segment"]))
    print("  trial start times monotonic:", bool(np.all(np.diff(d["start_time"]) > 0)))
    print("  -500..+1000 ms window inside the trial interval for", int(S["info"]["ok_window"].sum()), "of",
          len(S["info"]["ok_window"]), "kept trials")
    print("  delay (go - target) s: range", np.round(np.nanpercentile(d["delay"], [0, 50, 100]), 3))
    print("  movement onset - go cue s: range", np.round(np.nanpercentile(d["move_begins_time"] - d["go_cue_time"], [0, 50, 100]), 3))
    h = S["hand"]
    print("  hand position (file unit 'meters'): range x", np.round(np.nanmin(h[..., 0]), 1), np.round(np.nanmax(h[..., 0]), 1),
          "y", np.round(np.nanmin(h[..., 1]), 1), np.round(np.nanmax(h[..., 1]), 1), "| trials without hand",
          int(np.isnan(h[:, 0, 0]).sum()))
    sp_hand = np.hypot(*np.gradient(h, axis=1).transpose(2, 0, 1)) * 1000
    print("  peak hand speed per trial (file units per s): median", np.round(np.nanmedian(np.nanmax(sp_hand, 1)), 1))
    print("  reach duration ms: median", np.nanmedian(tr["duration_ms"]), "range", np.nanmin(tr["duration_ms"]), np.nanmax(tr["duration_ms"]))
    print("  smoothed rates (spikes/s): range", float(S["R"].min()), float(S["R"].max()), "mean", float(S["R"].mean()))
print("peak memory GB", resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6)

### Conditions and trial readings

The claims file gives two readings for the 72 conditions (drop maze conditions that are a multiple of 3;
drop the 3-target versions, whose distractors repeat a curved maze) and two for trials (all successful
trials; successful trials with delay above 400 ms). Analyses below use trial reading (b), the one whose
counts are closer to the paper's (assumption, recorded).

In [ ]:
CT = {}
for m in ("J", "N"):
    d = SESS[m]["all_trials"]
    mc = d["maze_condition"]
    a = np.unique(mc[(mc % 3 != 0)])
    b = np.unique(mc[(d["maze_num_targets"] == 1)])
    print(m, "reading (a) conditions", len(a), "| reading (b) 1-target conditions", len(b),
          "| (b) minus maze 0:", len(np.setdiff1d(b, [0])), "| same set as (a):", np.array_equal(np.setdiff1d(b, [0]), a))
    print("   maze 0 trials:", int((mc == 0).sum()), "targets", np.unique(d["maze_num_targets"][mc == 0]),
          "barriers", np.unique(d["maze_num_barriers"][mc == 0]), "novel_maze", np.unique(d["novel_maze"][mc == 0]))
    tr = SESS[m]["trials"]
    CT[m] = ldr.condition_table(tr)
    st = CT[m]["straight"]
    print("   conditions kept:", len(CT[m]["maze"]), "| straight (no barrier):", int(st.sum()), "| curved:", int((~st).sum()))
    for r in ("a", "b"):
        mk = ldr.select_trials(tr, r)
        print(f"   trial reading ({r}): {int(mk.sum())} trials; per condition {np.bincount(CT[m]['cond'][mk]).min()} to {np.bincount(CT[m]['cond'][mk]).max()}")
    print("   with correct_reach also required (not a reading):", int((ldr.select_trials(tr, 'b') & (tr['correct_reach'] == 1)).sum()))
MASK = {m: ldr.select_trials(SESS[m]["trials"], "b") & np.isfinite(SESS[m]["hand"][:, 0, 0]) for m in ("J", "N")}
print({m: int(MASK[m].sum()) for m in MASK})

### Clock check

Spikes and the trial table must share a clock. For each recording segment, the condition of a trial is
decoded from spike counts in -100 to +300 ms around movement onset (nearest centroid, leave one trial
out; chance 1/72). As a contrast, the same decoder is run with each trial's activity paired with the
condition of the previous trial in the segment.

In [ ]:
def clock_check(m):
    S = SESS[m]
    tr = S["trials"]
    cond = CT[m]["cond"]
    gi = (ldr.GRID_MS >= -100) & (ldr.GRID_MS <= 300)
    F = S["R"][:, :, gi].mean(2)
    out = []
    for sg in np.unique(tr["segment"]):
        ii = np.where((tr["segment"] == sg) & MASK[m])[0]
        res = []
        for lab in (cond[ii], np.r_[cond[ii][-1], cond[ii][:-1]]):
            X = F[ii]
            hits = 0
            for q in range(len(ii)):
                o = np.ones(len(ii), bool)
                o[q] = False
                cs = np.unique(lab[o])
                cen = np.array([X[o & (lab == c)].mean(0) for c in cs])
                hits += cs[np.argmin(((cen - X[q]) ** 2).sum(1))] == lab[q]
            res.append(hits / len(ii))
        # trained on the other segments, tested on this one
        oth = np.where((tr["segment"] != sg) & MASK[m])[0]
        cross = np.nan
        if len(oth):
            cs = np.unique(cond[oth])
            cen = np.array([F[oth][cond[oth] == c].mean(0) for c in cs])
            pred = cs[np.argmin(((F[ii][:, None, :] - cen[None]) ** 2).sum(2), 1)]
            cross = (pred == cond[ii]).mean()
        out.append((sg, len(ii), res[0], res[1], cross))
    return out


for m in ("J", "N"):
    for sg, n, acc, acc_sh, cross in clock_check(m):
        print(f"{m} segment {sg}: {n} trials, condition decoded {acc:.3f} (labels of the previous trial {acc_sh:.3f}; "
              f"trained on the other segments {cross:.3f}; chance {1/72:.3f})")

## Basis functions per dataset

For each dataset, condition-averaged rates are formed from trial reading (b), and the number of temporal
basis functions is cross-validated (Methods, Cross-validating the number of temporal basis functions;
claims, shared method: k = 1 to 20, two halves of the trials, 100 folds, an even k raised by one). The
basis is fitted on -150 to +850 ms (Methods, De-mixing temporal basis functions) and de-mixed. The
cross-validation is repeated under the five seeds of fig3g-basis-frequencies.

In [ ]:
D = {}
for ds in DATASETS:
    m, area = ds
    S = SESS[m]
    u = S["area"] == area
    R = S["R"][:, u]
    cond = CT[m]["cond"]
    X = ldr.cond_average(R, cond, 72, MASK[m])
    D[ds] = dict(R=R, X=X, Xw=X[..., ldr.FIT0:], cond=cond, u=u)

def cv_job(args):
    ds, sd = args
    return ldr.cv_tbf_k(D[ds]["R"], D[ds]["cond"], 72, np.random.default_rng(sd), mask=MASK[ds[0]])


_jobs = [(ds, sd) for ds in DATASETS for sd in seeds5("fig3g-basis-frequencies")]
_cv = dict(zip(_jobs, ldr.pmap(cv_job, _jobs)))
CV = {ds: _cv[(ds, seed("fig3g-basis-frequencies"))] for ds in DATASETS}
for ds in DATASETS:
    ks = [int(np.argmax(_cv[(ds, sd)].mean(0)) + 1) for sd in seeds5("fig3g-basis-frequencies")]
    k = ks[0] + (ks[0] % 2 == 0)
    D[ds]["k_cv"] = ks[0]
    D[ds]["k_seeds"] = ks
    # de-mix; drop the last basis function while more than one real (unpaired) mode remains
    # (Methods: 'discarded the last basis function if it was unpaired')
    while True:
        B, L = ldr.tbf(D[ds]["Xw"], k)
        Bd, Ld, lam, grp, fr = ldr.demix(B, L)
        if (grp == -1).sum() <= 1 or k == 1:
            break
        k -= 1
    D[ds].update(k=k, B=Bd, L=Ld, lam=lam, grp=grp, freq=fr, Braw=B, Lraw=L)
    print(NAME[ds], "held-out VE by k:", np.round(CV[ds].mean(0)[:12], 3))
    print("   best k under five seeds", ks, "-> k used", k, "| frequencies (Hz)", np.round(fr, 2),
          "| |eigenvalue|", np.round(np.abs(lam), 3))

## data-composition

> Two monkeys (J and N) performed the maze task with 72 conditions (36 straight, 36 curved); a PMd and
> an M1 array in each monkey gave 79 to 118 units per array; 1469 trials for monkey J and 2014 for
> monkey N.

In [ ]:
for m, paper_n in (("J", 1469), ("N", 2014)):
    tr = SESS[m]["trials"]
    for r in ("a", "b"):
        n = int(ldr.select_trials(tr, r).sum())
        print(f"monkey {m}, trial reading ({r}): {n} trials (paper {paper_n}, tolerance {paper_n * 0.1:.1f}; "
              f"within: {abs(n - paper_n) <= paper_n * 0.1}; difference {n - paper_n:+d})")
    print(f"monkey {m}: conditions {len(CT[m]['maze'])}, straight {int(CT[m]['straight'].sum())}, curved {int((~CT[m]['straight']).sum())}")
    for a in ("M1", "PMd"):
        print(f"   {a}: {int((SESS[m]['area'] == a).sum())} units; units table columns hold no single/multi-unit label")
print("units per electrode (J):", np.unique(np.bincount(np.arange(192))))

**Result: partially reproduced.** Both monkeys have the 72 conditions (36 straight, 36 curved), and the trial counts are within 10% of the paper's under both readings (J 1595 or 1381 against 1469; N 2121 or 1884 against 2014). Each array holds 96 units, one per electrode, and the units table carries no single- or multi-unit label, so the count the criterion asks for (units the dataset marks as single or multi-units) cannot be made; 96 lies in 71 to 130, but these are not shown to be the paper's 79 to 118 sorted units.

## Single-condition LDSs: fig2c-condition-lds-variance

> Activity on each single condition was low-dimensional and approximated by an LDS of 5 to 9 dimensions,
> which explained 71 to 93% of the population variance.

Dimensionality by cross-validation over two trial partitions, 100 repeats, odd k from 1 to 21 (the
code's set; claims), one k per dataset. The Methods' even split is the main reading and the code's 3/4
to 1/4 split a variant.

In [ ]:
KS = list(range(1, 22, 2))


def dim_job(args):
    ds, sd, frac, folds = args
    cv = ldr.cv_condition_dim(D[ds]["R"], D[ds]["cond"], 72, np.random.default_rng(sd), KS, folds=folds, mask=MASK[ds[0]], frac=frac)
    return cv.mean((0, 1))


_jobs = [(ds, seed("fig2c-condition-lds-variance"), fr, 100) for ds in DATASETS for fr in (0.5, 0.75)]
_jobs += [(ds, sd, 0.5, 100) for ds in DATASETS for sd in seeds5("fig2c-condition-lds-variance")[1:]]
DIMCV = dict(zip(_jobs, ldr.pmap(dim_job, _jobs)))
for ds in DATASETS:
    cvm = DIMCV[(ds, seed("fig2c-condition-lds-variance"), 0.5, 100)]
    res = {"even split": KS[int(np.argmax(cvm))],
           "3/4 split": KS[int(np.argmax(DIMCV[(ds, seed("fig2c-condition-lds-variance"), 0.75, 100)]))]}
    kd = res["even split"]
    sims, lams = [], []
    for c in range(72):
        Xc = D[ds]["Xw"][c]
        Xs, lam = ldr.rrr_lds(Xc, kd)
        sims.append(ldr.pop_ve(Xc, Xs))
        lams.append(lam)
    D[ds]["lds_k"] = kd
    D[ds]["lds_ve"] = np.array(sims)
    D[ds]["lds_lam"] = lams
    kv = res["3/4 split"]
    ve_v = np.array([ldr.pop_ve(D[ds]["Xw"][c], ldr.rrr_lds(D[ds]["Xw"][c], kv)[0]) for c in range(72)])
    D[ds]["lds_ve_var"] = ve_v
    print(f"{NAME[ds]}: held-out VE by k {dict(zip(KS, np.round(cvm, 3)))}")
    print(f"   chosen k {kd} (even split) | {kv} (3/4 split) ; LDS VE mean {np.mean(sims):.3f} s.d. {np.std(sims):.3f} "
          f"range {np.min(sims):.3f}-{np.max(sims):.3f} | at k={kv}: mean {ve_v.mean():.3f}")
# seed sensitivity of the chosen k
for ds in DATASETS:
    ks = [KS[int(np.argmax(DIMCV[(ds, sd, 0.5, 100)]))] for sd in seeds5("fig2c-condition-lds-variance")[1:]]
    print(NAME[ds], "chosen k under the four other seeds:", ks)

**Result: reproduced.** The cross-validated dimensionality is 5, 5, 7 and 5 (M1-J, PMd-J, M1-N, PMd-N), the same under all five seeds, and the single-condition LDSs explain on average 84.4, 90.2, 94.2 and 90.6% of the population variance, all within 63.9 to 100%. The values are at or above the top of the paper's 71 to 93%. The code's 3/4 split chooses 7, 5, 7 and 7.

## fig3a-eigenvalue-clusters

> Eigenvalues of the single-condition LDSs formed distinct clusters, corresponding to rotations at about
> 0.5, 1.5, 2.5 and 4 Hz on every condition, along with an offset.

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(14, 3.2))
for ax, ds in zip(axs, DATASETS):
    lams = D[ds]["lds_lam"]
    pairs = [np.sort(np.abs(np.angle(l[l.imag > 1e-12]))) / (2 * np.pi) * 100 for l in lams]
    npair = np.array([len(p) for p in pairs])
    nreal = np.array([int(np.sum(np.abs(l.imag) <= 1e-12)) for l in lams])
    maxp = npair.max()
    med = [np.median([p[i] for p in pairs if len(p) > i]) for i in range(maxp)]
    cnt = [sum(len(p) > i for p in pairs) for i in range(maxp)]
    targets = [0.5, 1.5, 2.5, 4.0]
    ok = [abs(med[i] - targets[i]) <= 0.1 * targets[i] for i in range(min(maxp, 4))]
    print(f"{NAME[ds]}: k={D[ds]['lds_k']}; complex pairs per condition {np.bincount(npair)} ; real eigenvalues per condition "
          f"{np.bincount(nreal)} ; conditions with >=1 real: {int((nreal >= 1).sum())}/72")
    print(f"   median frequency of pair rank 1..{maxp} (Hz): {np.round(med, 2)} (conditions: {cnt}); within 10% of 0.5,1.5,2.5,4: {ok}")
    allf = np.concatenate([np.abs(np.angle(l)) / (2 * np.pi) * 100 for l in lams])
    print(f"   all eigenvalue frequencies: range {allf.min():.2f}-{allf.max():.2f} Hz; |eigenvalue| range "
          f"{min(np.abs(l).min() for l in lams):.3f}-{max(np.abs(l).max() for l in lams):.3f}")
    L_ = np.concatenate(lams)
    ax.scatter(L_.real, L_.imag, s=3)
    ax.set_title(NAME[ds]); ax.set_xlabel("real"); ax.set_ylabel("imag")
    ax.set_xlim(0.8, 1.01); ax.set_ylim(-0.25, 0.25)
    print(f"   points outside the axes: {int(np.sum((L_.real < 0.8) | (np.abs(L_.imag) > 0.25)))}")
plt.tight_layout(); plt.show()

**Result: partially reproduced.** Every condition's LDS has the same number of complex pairs (2; 3 in M1-N, where one condition has 2 pairs and 3 real eigenvalues) and at least one real eigenvalue, so the eigenvalues form distinct clusters with an offset. Only PMd-J has its median frequencies within 10% of the paper's (0.54 and 1.53 Hz); M1-J has 0.85 and 2.35 Hz, M1-N 0.68, 1.74 and 3.21 Hz, PMd-N 0.73 and 1.67 Hz.

## fig3c-eigenvalue-distance-auc

> Eigenvalues differed between conditions only slightly more than expected from estimation noise, with
> ROC-AUC 0.51 to 0.58 between the between-condition distances and the noise distribution.

Distance: mean over frequency-sorted corresponding eigenvalues of their distance in the complex plane
(claims); the code's vector norm is a variant. Nulls: (a) Poisson, two sets of 20 trials simulated from
each condition's rates; (b) random halves of the real trials. The Methods give 500 repeats per
condition; 100 are run here for each reading (deviation; time). The Poisson trials are simulated as
10 ms counts from the smoothed rates and smoothed with a 20 ms Gaussian at 10 ms resolution
(deviation: the paper simulates at the resolution of its 1 ms pipeline).

In [ ]:
def eig_dist(l1, l2, how="mean"):
    a, b = ldr.sort_eigs(l1), ldr.sort_eigs(l2)
    n = min(len(a), len(b))
    if how == "mean":
        return np.mean(np.abs(a[:n] - b[:n]))
    return np.linalg.norm(a[:n] - b[:n])


def eig_null(ds, reading, nrep, sd):
    rng = np.random.default_rng(sd)
    m = ds[0]
    kd = D[ds]["lds_k"]
    R, cond = D[ds]["R"], D[ds]["cond"]
    out_mean, out_norm = [], []
    if reading == "b":
        for r in range(nrep):
            a, b = ldr.split_half(cond, 72, rng, MASK[m])
            XA = ldr.cond_average(R, cond, 72, a)[..., ldr.FIT0:]
            XB = ldr.cond_average(R, cond, 72, b)[..., ldr.FIT0:]
            for c in range(72):
                la, lb = ldr.rrr_lds(XA[c], kd)[1], ldr.rrr_lds(XB[c], kd)[1]
                out_mean.append(eig_dist(la, lb)); out_norm.append(eig_dist(la, lb, "norm"))
    else:
        from scipy.ndimage import gaussian_filter1d
        lam_rate = np.clip(D[ds]["X"], 0, None) * 0.01 * 20  # expected count of 20 trials per 10 ms bin
        for r in range(nrep):
            for c in range(72):
                est = []
                for _ in range(2):
                    cnt = rng.poisson(lam_rate[c]).astype(float)
                    est.append(gaussian_filter1d(cnt, 2.0, axis=1, mode="reflect")[:, ldr.FIT0:] / (0.01 * 20))
                la, lb = ldr.rrr_lds(est[0], kd)[1], ldr.rrr_lds(est[1], kd)[1]
                out_mean.append(eig_dist(la, lb)); out_norm.append(eig_dist(la, lb, "norm"))
    return np.array(out_mean), np.array(out_norm)


def eig_job(args):
    return eig_null(*args)


_jobs = [(ds, r, 100, seed("fig3c-eigenvalue-distance-auc")) for ds in DATASETS for r in ("a", "b")]
_jobs += [(ds, "b", 20, sd) for ds in DATASETS for sd in seeds5("fig3c-eigenvalue-distance-auc")[1:]]
_res = dict(zip(_jobs, ldr.pmap(eig_job, _jobs)))
EIG = {}
for ds in DATASETS:
    lams = D[ds]["lds_lam"]
    obs = np.array([eig_dist(lams[i], lams[j]) for i in range(72) for j in range(i + 1, 72)])
    obs_n = np.array([eig_dist(lams[i], lams[j], "norm") for i in range(72) for j in range(i + 1, 72)])
    EIG[ds] = dict(obs=obs, obs_n=obs_n)
    for r in ("a", "b"):
        EIG[ds][r] = _res[(ds, r, 100, seed("fig3c-eigenvalue-distance-auc"))]
    EIG[ds]["seeds"] = [_res[(ds, "b", 20, sd)][0] for sd in seeds5("fig3c-eigenvalue-distance-auc")[1:]]

In [ ]:
for ds in DATASETS:
    e = EIG[ds]
    for reading in ("a", "b"):
        nm, nn = e[reading]
        A, raw = ldr.auc(e["obs"], nm)
        An, rawn = ldr.auc(e["obs_n"], nn)
        print(f"{NAME[ds]} null ({reading}): AUC {A:.3f} (unfolded {raw:.3f}); median distance between conditions "
              f"{np.median(e['obs']):.4f}, noise {np.median(nm):.4f} | code's norm variant AUC {An:.3f}")
# seed sensitivity, null (b), 20 repeats under each of the four other seeds
for ds in DATASETS:
    vals = [ldr.auc(EIG[ds]["obs"], nm)[0] for nm in EIG[ds]["seeds"]]
    print(NAME[ds], "null (b) AUC under four other seeds (20 repeats each):", np.round(vals, 3))

**Result: not reproduced.** The AUC between distances across conditions and the noise distribution is 0.81, 0.71, 0.63 and 0.70 under the Poisson null (a) and 0.70, 0.64, 0.66 and 0.68 under the half-split null (b), against 0.459 to 0.638; only M1-N under (a) is in range. Eigenvalues differ between conditions more than the paper reports, though the distances are small (median 0.011 to 0.016 in the complex plane). The five seeds give AUCs within 0.02 of each other under null (b).

## suppfig2-eigenvalues-from-kinematics

This claim needs the reach parameters of fig6e; it is computed in the kinematics section below.

## fig3g-basis-frequencies

> The cross-validated temporal basis functions contained 3 to 4 rotations at 0.5, 1.5, 2.5 and 4 Hz,
> along with an offset.

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(14, 3))
for ax, ds in zip(axs, DATASETS):
    g, fr = D[ds]["grp"], D[ds]["freq"]
    rot_f = [fr[g == r][0] for r in range(g.max() + 1)]
    near = [min([0.5, 1.5, 2.5, 4.0], key=lambda t: abs(t - f)) for f in rot_f]
    within = [abs(f - t) <= 0.1 * t for f, t in zip(rot_f, near)]
    print(f"{NAME[ds]}: CV best k {D[ds]['k_cv']} (five seeds {D[ds]['k_seeds']}), k used {D[ds]['k']}; rotation pairs {g.max() + 1}, "
          f"real modes {(g == -1).sum()}; rotation frequencies {np.round(rot_f, 2)} Hz; nearest of 0.5/1.5/2.5/4 {near}; within 10%: {within}")
    print(f"   |eigenvalue| of the offset mode {np.abs(D[ds]['lam'][g == -1]).round(4)}")
    ax.plot(ldr.GRID_MS[ldr.FIT0:], D[ds]["B"].T)
    ax.set_title(NAME[ds]); ax.set_xlabel("ms from movement onset")
plt.tight_layout(); plt.show()

**Result: partially reproduced.** The cross-validated factorization has 3 rotation pairs and one offset in M1-J, M1-N and PMd-N, and 2 pairs in PMd-J. The rotation frequencies are 0.70, 1.82, 3.08 (M1-J), 0.64, 1.54 (PMd-J), 0.69, 1.79, 3.23 (M1-N) and 0.74, 1.55, 2.71 Hz (PMd-N); only 3 of the 11 are within 10% of 0.5, 1.5, 2.5 or 4 Hz, and no dataset has a rotation near 0.5 Hz or 4 Hz.

## tbf-variance-explained

> The shared rotations of the temporal basis factorization explained 90 to 96% of the population variance.

In [ ]:
for ds in DATASETS:
    Xw = D[ds]["Xw"]
    ve_a = ldr.pop_ve(Xw, np.einsum("cnk,kt->cnt", D[ds]["L"], D[ds]["B"]))
    B2, L2 = ldr.tbf(D[ds]["X"], D[ds]["k"])
    ve_b = ldr.pop_ve(D[ds]["X"], np.einsum("cnk,kt->cnt", L2, B2))
    D[ds]["tbf_ve"] = ve_a
    print(f"{NAME[ds]}: (a) -150..+850 ms, de-mixed basis: mean {ve_a.mean():.3f} s.d. {ve_a.std():.3f} range {ve_a.min():.3f}-{ve_a.max():.3f} | "
          f"(b) -350..+850 ms, raw SVD basis: mean {ve_b.mean():.3f} s.d. {ve_b.std():.3f}")

**Result: reproduced.** The factorization explains on average 93.5, 93.6, 96.6 and 95.3% of the population variance (s.d. 0.8 to 1.5% across conditions) over -150 to +850 ms, and 92.1 to 96.1% over the whole window with the raw basis, all within 81 to 100%.

## tbf-shuffle-control and fig3h-recoverability

Shuffled data: the 1 ms bins of each trial permuted in time (one permutation for all units of a trial)
before smoothing and averaging, then TBF with the dataset's k. Recoverability per condition: the
variance of the basis functions explained by pinv(L(c)) X(c) (Methods, Eq. 6).

In [ ]:
def recoverability(L, B, Xw):
    out = []
    for c in range(len(L)):
        Bh = np.linalg.pinv(L[c]) @ Xw[c]
        out.append(ldr.pop_ve(B, Bh))
    return np.array(out)


SH = {}
for ds in DATASETS:
    m, area = ds
    D[ds]["rec"] = recoverability(D[ds]["L"], D[ds]["B"], D[ds]["Xw"])
    SH[ds] = {}
    for sd in seeds5("tbf-shuffle-control"):
        Xs = ldr.cond_average(SESS[m]["RS"][sd][:, D[ds]["u"]], D[ds]["cond"], 72, MASK[m])[..., ldr.FIT0:]
        Bs, Ls = ldr.tbf(Xs, D[ds]["k"])
        ve = ldr.pop_ve(Xs, np.einsum("cnk,kt->cnt", Ls, Bs))
        SH[ds][sd] = dict(ve=ve, rec=recoverability(Ls, Bs, Xs))

In [ ]:
s0 = seed("tbf-shuffle-control")
for ds in DATASETS:
    real, sh = D[ds]["tbf_ve"], SH[ds][s0]["ve"]
    p_rs = stats.ranksums(real, sh).pvalue
    p_sr = stats.wilcoxon(real, sh).pvalue
    print(f"{NAME[ds]}: shuffled VE mean {sh.mean():.3f} s.d. {sh.std():.3f} (real {real.mean():.3f}); rank-sum p {p_rs:.2e}; "
          f"signed-rank (variant) p {p_sr:.2e}; real higher on {int((real > sh).sum())}/72")
    print("   shuffled mean under the five seeds:", [round(SH[ds][sd]["ve"].mean(), 4) for sd in seeds5("tbf-shuffle-control")])

**Result: partially reproduced.** Shuffling time bins lowers the variance explained to 42.2, 28.7, 41.8 and 42.1%, below the real data on all 72 conditions of every dataset (rank-sum p = 3.9e-25). Three of four lie in 31.5 to 50.6%; PMd-J (28.7%) is below. The five seeds move the shuffled means by less than 0.01.

In [ ]:
for ds in DATASETS:
    rec, ch = D[ds]["rec"], SH[ds][s0]["rec"]
    print(f"{NAME[ds]}: recoverability mean {rec.mean():.3f} s.d. {rec.std():.3f}; chance (shuffled data) mean {ch.mean():.3f}; "
          f"rank-sum p {stats.ranksums(rec, ch).pvalue:.2e}; real higher on {int((rec > ch).sum())}/72")
    print("   chance mean under the five seeds:", [round(SH[ds][sd]["rec"].mean(), 4) for sd in seeds5("tbf-shuffle-control")])

**Result: partially reproduced.** Recoverability is 97.3, 97.9, 98.6 and 98.6%, within 81.9 to 100%, and above the shuffled data on every condition (rank-sum p = 3.9e-25). The chance level is higher than the paper's: 89.6, 86.0, 89.0 and 89.6%, so only PMd-J lies in 56.7 to 88%.

## fig3i-duration-warping

Reach duration per trial: movement onset to the last 1 ms sample at which hand speed is at least a tenth
of its peak, within 850 ms (claims; code warpTrials.m). Per condition, the mean over its trials;
correlation of |duration - mean over conditions| with recoverability, before and after warping.

In [ ]:
for ds in DATASETS:
    m = ds[0]
    dur = SESS[m]["trials"]["duration_ms"]
    cdur = np.array([np.nanmean(dur[MASK[m] & (D[ds]["cond"] == c)]) for c in range(72)])
    ext = np.abs(cdur - cdur.mean())
    D[ds]["cdur"] = cdur
    r0, p0 = stats.pearsonr(ext, D[ds]["rec"])
    out = [f"{NAME[ds]}: condition duration {cdur.min():.0f}-{cdur.max():.0f} ms; unwarped rho {r0:.3f} p {p0:.3g}"]
    for key, lab in (("RW", "warped, truncated at 850 (main)"), ("RW2", "warped, post-reach to 250 ms")):
        Xwp = ldr.cond_average(SESS[m][key][:, D[ds]["u"]], D[ds]["cond"], 72, MASK[m])[..., ldr.FIT0:]
        Bw, Lw = ldr.tbf(Xwp, D[ds]["k"])
        Bw, Lw = ldr.demix(Bw, Lw)[:2]
        recw = recoverability(Lw, Bw, Xwp)
        r, p = stats.pearsonr(ext, recw)
        out.append(f"{lab}: rho {r:.3f} p {p:.3g} (recoverability mean {recw.mean():.3f})")
    print(" | ".join(out))

**Result: not reproduced.** Before warping, M1-N has rho = +0.235 (p = 0.047), opposite in sign to the paper's -0.26; the other three (-0.157, 0.083, 0.077) lie in the stated range. Warping to 600 ms induces a negative correlation only in M1-J (rho = -0.514, p = 4e-06); PMd-J, M1-N and PMd-N give -0.071, -0.024 and -0.193 (p > 0.1). The 250 ms post-reach variant gives -0.36, -0.18, -0.10 and -0.32.

## single-lds-variance

> A single linear dynamical system fitted to all conditions captured on average 46 to 66% of the
> population variance.

In [ ]:
for ds in DATASETS:
    X = D[ds]["X"]
    Xp = np.concatenate([X[c, :, ldr.FIT0:-1].T for c in range(72)])
    Xf = np.concatenate([X[c, :, ldr.FIT0 + 1:].T for c in range(72)])
    M = np.linalg.lstsq(Xp, Xf, rcond=None)[0]
    ve_a, ve_b = [], []
    for c in range(72):
        sim = np.zeros((X.shape[2], X.shape[1]))
        sim[: ldr.FIT0 + 1] = X[c, :, : ldr.FIT0 + 1].T
        for t in range(ldr.FIT0 + 1, X.shape[2]):
            sim[t] = sim[t - 1] @ M
        ve_a.append(ldr.pop_ve(X[c, :, ldr.FIT0:], sim[ldr.FIT0:].T))
        ve_b.append(ldr.pop_ve(X[c], sim.T))
    ve_a, ve_b = np.array(ve_a), np.array(ve_b)
    D[ds]["slds_ve"] = ve_a
    print(f"{NAME[ds]}: (a) -150..+850 mean {ve_a.mean():.3f} s.d. {ve_a.std():.3f} | (b) whole window, bins before -150 copied "
          f"mean {ve_b.mean():.3f} s.d. {ve_b.std():.3f} | max |eigenvalue| of M {np.abs(np.linalg.eigvals(M)).max():.4f}")

**Result: not reproduced.** One LDS fitted to all conditions explains on average 76.6, 80.5, 82.1 and 77.4% of the population variance over -150 to +850 ms (79 to 87% over the whole window), above the range 41.4 to 72.6% in every dataset.

## Alignment of rotations across conditions

Rotations are the pairs of de-mixed basis functions that share a frequency; the state space location
is the column of the real (offset) mode. Noise: loadings fitted by regression on the fixed basis to
two random halves of a condition's trials, 36 repeats (Methods, Measuring the across-condition
alignment of rotations).

In [ ]:
def comps(ds):
    g = D[ds]["grp"]
    out = [("rot%d (%.2f Hz)" % (r, D[ds]["freq"][g == r][0]), np.where(g == r)[0]) for r in range(g.max() + 1)]
    out.append(("location", np.where(g == -1)[0][:1]))
    return out


def ai_matrix(L1, B1, L2):
    """AI of each condition's activity L1[c] @ B1 onto span(L2[c']) for all c, c'. Returns (C, C)."""
    C = len(L1)
    Q = np.stack([ldr.orth(L2[c]) for c in range(C)])
    out = np.zeros((C, C))
    for c in range(C):
        Y = L1[c] @ B1
        Y = Y - Y.mean(1, keepdims=True)
        out[c] = np.sum(np.einsum("knd,nt->kdt", Q, Y) ** 2, axis=(1, 2)) / np.sum(Y ** 2)
    return out


def noise_ai(ds, nrep, sd):
    rng = np.random.default_rng(sd)
    m = ds[0]
    Bp = np.linalg.pinv(D[ds]["B"])
    out = {name: [] for name, _ in comps(ds)}
    halves = []
    for r in range(nrep):
        a, b = ldr.split_half(D[ds]["cond"], 72, rng, MASK[m])
        LA = ldr.cond_average(D[ds]["R"], D[ds]["cond"], 72, a)[..., ldr.FIT0:] @ Bp
        LB = ldr.cond_average(D[ds]["R"], D[ds]["cond"], 72, b)[..., ldr.FIT0:] @ Bp
        halves.append((LA, LB))
        for name, idx in comps(ds):
            out[name].append([ldr.alignment_index(LA[c][:, idx], D[ds]["B"][idx], LB[c][:, idx]) for c in range(72)])
    return {k: np.array(v) for k, v in out.items()}, halves


AL = {}
for ds in DATASETS:
    L, B = D[ds]["L"], D[ds]["B"]
    nz, halves = noise_ai(ds, 36, seed("fig4b-plane-alignment"))
    AL[ds] = dict(noise=nz, halves=halves, obs={})
    for name, idx in comps(ds):
        AL[ds]["obs"][name] = ai_matrix(L[:, :, idx], B[idx], L[:, :, idx])
    AL[ds]["noise_seeds"] = [noise_ai(ds, 36, sd)[0] for sd in seeds5("fig4b-plane-alignment")[1:]]


def align_report(ds, name):
    O = AL[ds]["obs"][name]
    off = ~np.eye(72, dtype=bool)
    obs = O[off]
    nz = AL[ds]["noise"][name].ravel()
    A, raw = ldr.auc(nz, obs)  # raw: P(noise > observed)
    per_c_obs = np.array([O[c][off[c]].mean() for c in range(72)])
    per_c_nz = AL[ds]["noise"][name].mean(0)
    p_a = stats.wilcoxon(per_c_obs, per_c_nz).pvalue
    p_b = stats.ranksums(obs, nz).pvalue
    seeds_auc = [ldr.auc(ns[name].ravel(), obs)[0] for ns in AL[ds]["noise_seeds"]]
    return A, raw, obs, nz, p_a, p_b, per_c_obs, per_c_nz, seeds_auc

## fig4b-plane-alignment

> Same-frequency rotational planes on pairs of conditions were far less aligned than expected from
> estimation noise (ROC-AUC 0.85 to 0.96).

In [ ]:
for ds in DATASETS:
    for name, idx in comps(ds)[:-1]:
        A, raw, obs, nz, p_a, p_b, pco, pcn, sa = align_report(ds, name)
        print(f"{NAME[ds]} {name}: AUC {A:.3f} (P(noise > observed) {raw:.3f}); observed median {np.median(obs):.3f}, "
              f"noise median {np.median(nz):.3f}; (a) signed-rank over conditions p {p_a:.2e} (obs lower on "
              f"{int((pco < pcn).sum())}/72); (b) rank-sum p {p_b:.2e}; AUC under four other seeds {np.round(sa, 3)}")

**Result: reproduced.** For every rotation in every dataset, same-frequency planes on pairs of conditions are less aligned than half-split estimates of the same condition: AUC 0.861 to 0.958, observed alignment lower on 72 of 72 conditions, signed-rank p = 1.7e-13 (reading a) and rank-sum p printed as 0 (reading b). The four other seeds change the AUCs by at most 0.002.

## suppfig6a-location-alignment

> The state space location occupied different dimensions across conditions, less aligned than expected
> from estimation noise (ROC-AUC 0.88 to 0.95).

In [ ]:
for ds in DATASETS:
    A, raw, obs, nz, p_a, p_b, pco, pcn, sa = align_report(ds, "location")
    print(f"{NAME[ds]} location: AUC {A:.3f} (P(noise > observed) {raw:.3f}); observed median {np.median(obs):.3f}, noise "
          f"median {np.median(nz):.3f}; (a) signed-rank p {p_a:.2e} (obs lower on {int((pco < pcn).sum())}/72); (b) rank-sum "
          f"p {p_b:.2e}; AUC under four other seeds {np.round(sa, 3)}")

**Result: partially reproduced.** The state space location is less aligned across conditions than expected from noise in every dataset (signed-rank p at most 2.1e-13), with AUC 0.883, 0.748, 0.870 and 0.945; PMd-J (0.748) is below 0.792.

## fig4c-subspace-excursion-angles

> Across conditions, each rotation's planes spanned 6 to 20 distinct dimensions with angles above 45
> degrees to each other, and more that varied above chance.

SEA to 25 dimensions, every condition tried as the seed. Excursion angles are counted after the seed
plane; the count including the seed's dimensions is shown as a variant. Chance: the largest principal
angle between the two half-split estimates of the same condition's plane (36 repeats above); an
excursion angle is above chance when it exceeds the 95th percentile of these (assumption: the code's
singleFrequencyOverlapControl.m is described in the claims only as the fraction of noise SEA above the
data).

In [ ]:
def max_angle(Q1, Q2):
    s = np.linalg.svd(Q1.T @ Q2, compute_uv=False)
    return np.degrees(np.arccos(np.clip(s.min(), -1, 1)))


def sea_job(args):
    ds, name = args
    idx = dict(comps(ds))[name]
    planes = [ldr.orth(D[ds]["L"][c][:, idx]) for c in range(72)]
    return ldr.sea(planes, 25)


jobs = [(ds, name) for ds in DATASETS for name, _ in comps(ds)]
SEA = dict(zip(jobs, ldr.pmap(sea_job, jobs)))
SEA_NOISE = {}
for ds in DATASETS:
    for name, idx in comps(ds):
        SEA_NOISE[(ds, name)] = np.array([max_angle(ldr.orth(LA[c][:, idx]), ldr.orth(LB[c][:, idx]))
                                          for LA, LB in AL[ds]["halves"] for c in range(72)])

In [ ]:
for ds in DATASETS:
    for name, idx in comps(ds)[:-1]:
        a = SEA[(ds, name)]
        thr = np.percentile(SEA_NOISE[(ds, name)], 95)
        print(f"{NAME[ds]} {name}: angles >45 deg: {int((a > 45).sum())} (with the 2 seed dimensions {int((a > 45).sum()) + 2}); "
              f"above chance ({thr:.1f} deg): {int((a > thr).sum())} of 25; angles {np.round(a[:12], 1)} ...")

**Result: partially reproduced.** The number of excursion angles above 45 degrees is 5, 13, 11 (M1-J), 7, 14 (PMd-J), 3, 9, 11 (M1-N) and 7, 18, 25 (PMd-N); 8 of 11 lie in 5.4 to 22 (M1-J 0.70 Hz, M1-N 0.69 Hz and PMd-N 2.71 Hz do not; counting the seed's two dimensions, 9 of 11 do). The number of angles above the half-split noise level is 4 to 9, fewer than those above 45 degrees for 7 of 11 rotations, so "far more that varied above chance" is not reproduced with the noise level used here, which is an assumption.

## rotation-orientation-dimensionality and suppfig6b-location-dimensionality

PCs to 80% and participation ratio from a PCA across conditions of (a) the loading columns stacked as
observations, (b) the activity of the component (loading columns times basis functions). SNR
dimensionality by the half-split procedure of the Methods, 100 folds: basis from partition 1, loadings
regressed on it for both partitions, PCA of partition 1's loadings; the dimensionality that maximises
the alignment index between the bottlenecked partition-1 loadings and partition 2's (all dimensions
searched).

In [ ]:
def pca_dims(Z):
    Z = Z - Z.mean(0)
    ev = np.linalg.svd(Z, compute_uv=False) ** 2
    cum = np.cumsum(ev) / ev.sum()
    return int(np.searchsorted(cum, 0.8) + 1), ldr.participation_ratio(ev)


def snr_job(args):
    ds, sd, nfold, dmax = args
    rng = np.random.default_rng(sd)
    m = ds[0]
    names = [n for n, _ in comps(ds)]
    full_f = {n: (D[ds]["freq"][i][0] if n != "location" else 0.0) for n, i in comps(ds)}
    score = {n: np.zeros(dmax) for n in names}
    for f_ in range(nfold):
        a, b = ldr.split_half(D[ds]["cond"], 72, rng, MASK[m])
        XA = ldr.cond_average(D[ds]["R"], D[ds]["cond"], 72, a)[..., ldr.FIT0:]
        XB = ldr.cond_average(D[ds]["R"], D[ds]["cond"], 72, b)[..., ldr.FIT0:]
        B1, L1 = ldr.tbf(XA, D[ds]["k"])
        B1, L1, lam1, g1, fr1 = ldr.demix(B1, L1)
        L2 = XB @ np.linalg.pinv(B1)
        for n in names:
            if n == "location":
                idx = np.where(g1 == -1)[0][:1]
            else:
                rs = [r for r in range(g1.max() + 1)]
                if not rs:
                    continue
                r = min(rs, key=lambda r: abs(fr1[g1 == r][0] - full_f[n]))
                idx = np.where(g1 == r)[0]
            A1, A2 = L1[:, :, idx], L2[:, :, idx]
            Z = A1.transpose(0, 2, 1).reshape(-1, A1.shape[1])
            V = np.linalg.svd(Z - Z.mean(0), full_matrices=False)[2]  # PCs of partition 1 (rows)
            Bc = B1[idx] - B1[idx].mean(1, keepdims=True)
            G = Bc @ Bc.T
            Q2 = np.stack([ldr.orth(A2[c]) for c in range(72)])
            C1 = np.einsum("pn,cnk->cpk", V, A1)  # loadings in PC coordinates
            W2 = np.einsum("cnd,pn->cdp", Q2, V)
            # bottleneck to d PCs: A1d = V[:d].T C1[:, :d]; the AI of A1d @ Bc onto span(Q2), for all d
            Mcum = np.cumsum(np.einsum("cdp,cpk->cpdk", W2, C1), axis=1)  # (c, d, dq, k)
            Ccum = np.cumsum(np.einsum("cpk,cpl->cpkl", C1, C1), axis=1)  # (c, d, k, k)
            num = np.einsum("cpdk,kl,cpdl->cp", Mcum, G, Mcum)
            den = np.einsum("cpkl,kl->cp", Ccum, G)
            ai = (num / den).mean(0)
            score[n][: min(dmax, len(ai))] += ai[:dmax] / nfold
    return score


SNR = dict(zip(DATASETS, ldr.pmap(snr_job, [(ds, seed("rotation-orientation-dimensionality"), 100, 96) for ds in DATASETS])))
_j = [(ds, sd, 100, 96) for ds in DATASETS for sd in seeds5("rotation-orientation-dimensionality")[1:]]
_r = ldr.pmap(snr_job, _j)
SNR_SEEDS = {ds: [r for (d_, *_), r in zip(_j, _r) if d_ == ds] for ds in DATASETS}

DIMS = {}
for ds in DATASETS:
    for name, idx in comps(ds):
        A = D[ds]["L"][:, :, idx]
        za = A.transpose(0, 2, 1).reshape(-1, A.shape[1])
        zb = np.concatenate([(A[c] @ D[ds]["B"][idx]).T for c in range(72)])
        pa, pra = pca_dims(za)
        pb, prb = pca_dims(zb)
        snr = int(np.argmax(SNR[ds][name]) + 1)
        snr_s = [int(np.argmax(s[name]) + 1) for s in SNR_SEEDS[ds]]
        DIMS[(ds, name)] = dict(pc_a=pa, pr_a=pra, pc_b=pb, pr_b=prb, snr=snr, snr_seeds=snr_s)

In [ ]:
for ds in DATASETS:
    for name, idx in comps(ds)[:-1]:
        d = DIMS[(ds, name)]
        print(f"{NAME[ds]} {name}: (a) PCs to 80% {d['pc_a']}, PR {d['pr_a']:.1f} | (b) PCs {d['pc_b']}, PR {d['pr_b']:.1f} | "
              f"SNR>1 dims {d['snr']} (four other seeds: {d['snr_seeds']})")

**Result: not reproduced.** Rotation orientations need 2 to 23 PCs to reach 80% of their variance (reading a; 2 to 22 under b) and have participation ratios of 1.8 to 18.7, against 20.7 to 44 and 15.03 to 40.48; only PMd-N 2.71 Hz is inside either range. The SNR dimensionality is 10 to 21 for 9 of 11 rotations, inside 6.3 to 36.3; for the slowest rotation of M1-N and PMd-N the alignment rises up to all 96 dimensions, so no optimum is found.

In [ ]:
for ds in DATASETS:
    d = DIMS[(ds, "location")]
    a = SEA[(ds, "location")]
    print(f"{NAME[ds]} location: SEA angles >15 deg {int((a > 15).sum())} (with the seed dimension {int((a > 15).sum()) + 1}); "
          f"(a) PCs to 80% {d['pc_a']}, PR {d['pr_a']:.1f} | (b) PCs {d['pc_b']}, PR {d['pr_b']:.1f} | SNR>1 dims {d['snr']} "
          f"(four other seeds: {d['snr_seeds']}); first angles {np.round(a[:8], 1)}")

**Result: not reproduced.** SEA finds 0 or 1 location dimensions with angles above 15 degrees (1 or 2 with the seed), the locations need 2 to 5 PCs for 80% of their variance, and their participation ratios are 2.3 to 4.2 (reading a), all far below the stated ranges. The SNR procedure reaches its maximum at the largest dimensionality tested (72), so it gives no estimate.

## fig5a-frequencies-segregated and suppfig6c-location-distinct-from-rotations

Alignment index of a component on one condition onto the same component, and onto each other
component, on a different condition. Reading (a): signed-rank over the same ordered condition pairs;
reading (b): the code's rank-sum test of the two distributions.

In [ ]:
def cross_ai(ds, n1, n2):
    i1, i2 = dict(comps(ds))[n1], dict(comps(ds))[n2]
    L, B = D[ds]["L"], D[ds]["B"]
    return ai_matrix(L[:, :, i1], B[i1], L[:, :, i2])


def seg_report(ds, n1, n2):
    off = ~np.eye(72, dtype=bool)
    same = AL[ds]["obs"][n1][off]
    diff = cross_ai(ds, n1, n2)[off]
    return same, diff, stats.wilcoxon(same, diff).pvalue, stats.ranksums(same, diff).pvalue

In [ ]:
for ds in DATASETS:
    names = [n for n, _ in comps(ds)][:-1]
    for n1 in names:
        for n2 in names:
            if n1 == n2:
                continue
            same, diff, pa, pb = seg_report(ds, n1, n2)
            print(f"{NAME[ds]} {n1} vs onto {n2}: same median {np.median(same):.3f}, different median {np.median(diff):.3f}; "
                  f"mean difference {np.mean(same - diff):+.3f}; same higher on {(same > diff).mean():.3f} of pairs; "
                  f"(a) p {pa:.1e}; (b) p {pb:.1e}")

**Result: partially reproduced.** Same-frequency alignment is higher than different-frequency alignment for 18 of the 20 rotation comparisons (medians 0.135 to 0.752 against 0.052 to 0.257). For M1-N 3.23 Hz onto the other two rotations, same-frequency alignment is not higher (median 0.176 against 0.191 and 0.193; higher on 49% and 48% of condition pairs). Their mean differences are positive (+0.038 and +0.042) and the signed-rank tests give p = 9e-11 and 4e-08, so the direction there depends on the measure; they are counted as not shown. In PMd-J 1.54 Hz onto 0.64 Hz and PMd-N 2.71 Hz onto the others the medians differ by 0.001, with mean differences of +0.033 to +0.036.

In [ ]:
for ds in DATASETS:
    for n2, _ in comps(ds)[:-1]:
        same, diff, pa, pb = seg_report(ds, "location", n2)
        print(f"{NAME[ds]} location vs onto {n2}: same median {np.median(same):.3f}, different median {np.median(diff):.3f}; "
              f"(a) p {pa:.1e}; (b) p {pb:.1e}")

**Result: reproduced.** In every dataset, the location on one condition is more aligned with the location on another (median 0.935 to 0.969) than with any rotation plane (median 0.078 to 0.670), with p printed as 0 under both readings.

## fig5c-rotations-predict-rotations and fig6b-location-predicts-rotations

Six folds over conditions; TBF, loadings and a ridge map fitted on 5/6 of the conditions; held-out
loadings by projecting the held-out activity on the training basis (claims; code
predictOneComponentFromAnother.m). Rotations in each fold are matched to the full-data rotations by
frequency (assumption).

In [ ]:
def fold_tbf(ds, train):
    Xtr = D[ds]["Xw"][train]
    B, L = ldr.tbf(Xtr, D[ds]["k"])
    B, L, lam, g, fr = ldr.demix(B, L)
    return B, g, fr


def match(ds, g, fr):
    out = {}
    for name, idx in comps(ds):
        if name == "location":
            out[name] = np.where(g == -1)[0][:1]
        else:
            f0 = D[ds]["freq"][idx][0]
            r = min(range(g.max() + 1), key=lambda r: abs(fr[g == r][0] - f0))
            out[name] = np.where(g == r)[0]
    return out


def svd_reduce(Xtr, frac):
    mu = Xtr.mean(0)
    U, S, Vt = np.linalg.svd(Xtr - mu, full_matrices=False)
    k = int(np.searchsorted(np.cumsum(S) / S.sum(), frac) + 1)
    return lambda Z: (Z - mu) @ Vt[:k].T


def predict_job(args):
    ds, sd, mode = args
    rng = np.random.default_rng(sd)
    folds = ldr.folds_of(72, 6 if mode != "loc_code" else 2, rng)
    names = [n for n, _ in comps(ds)]
    res = {}
    for te in folds:
        tr = np.setdiff1d(np.arange(72), te)
        B, g, fr = fold_tbf(ds, tr)
        mt = match(ds, g, fr)
        Bp = np.linalg.pinv(B)
        Ltr = D[ds]["Xw"][tr] @ Bp
        Lte = D[ds]["Xw"][te] @ Bp
        perm_tr, perm_te = rng.permutation(len(tr)), rng.permutation(len(te))
        if mode == "rot":
            pairs = [(a, b) for a in names[:-1] for b in names[:-1] if a != b]
            for a, b in pairs:
                Ptr = Ltr[:, :, mt[a]].reshape(len(tr), -1)
                Pte = Lte[:, :, mt[a]].reshape(len(te), -1)
                Ytr = Ltr[:, :, mt[b]].reshape(len(tr), -1)
                for null in (False, True):
                    red = svd_reduce(Ptr[perm_tr] if null else Ptr, 0.8)
                    f = ldr.ridge_fit(red(Ptr[perm_tr] if null else Ptr), Ytr, 0.1)
                    pred = f(red(Pte[perm_te] if null else Pte)).reshape(len(te), -1, 2)
                    for q, c in enumerate(te):
                        true = Lte[q][:, mt[b]] @ B[mt[b]]
                        res.setdefault((a, b, null), {})[c] = ldr.pop_ve(true, pred[q] @ B[mt[b]])
        else:
            loc = mt["location"]
            rest = np.setdiff1d(np.arange(B.shape[0]), loc)
            Ptr, Pte = Ltr[:, :, loc].reshape(len(tr), -1), Lte[:, :, loc].reshape(len(te), -1)
            Ytr = Ltr[:, :, rest].reshape(len(tr), -1)
            for null in (False, True):
                Pt = Ptr[perm_tr] if null else Ptr
                if mode == "loc":
                    f = ldr.ridge_fit(Pt, Ytr, 0.1)
                    pred = f(Pte[perm_te] if null else Pte)
                else:  # the code's variant: SVD to 90%, lambda 0.01, 2 folds
                    red = svd_reduce(Pt, 0.9)
                    f = ldr.ridge_fit(red(Pt), Ytr, 0.01)
                    pred = f(red(Pte[perm_te] if null else Pte))
                pred = pred.reshape(len(te), -1, len(rest))
                for q, c in enumerate(te):
                    Lh = np.zeros_like(Lte[q])
                    Lh[:, loc] = Lte[q][:, loc]
                    Lh[:, rest] = pred[q]
                    res.setdefault(("location", "rest", null), {})[c] = ldr.pop_ve(D[ds]["Xw"][c], Lh @ B)
    return {k: np.array([v[c] for c in range(72)]) for k, v in res.items()}


jobs = [(ds, sd, mode) for ds in DATASETS for mode in ("rot", "loc", "loc_code") for sd in seeds5("fig5c-rotations-predict-rotations")[:1]]
jobs += [(ds, sd, mode) for ds in DATASETS for mode in ("rot", "loc") for sd in seeds5("fig5c-rotations-predict-rotations")[1:]]
PRED = dict(zip(jobs, ldr.pmap(predict_job, jobs)))

In [ ]:
s0 = seed("fig5c-rotations-predict-rotations")
for ds in DATASETS:
    r = PRED[(ds, s0, "rot")]
    for (a, b, null), v in r.items():
        if null:
            continue
        nv = r[(a, b, True)]
        p = stats.wilcoxon(v, nv).pvalue
        p2 = stats.ranksums(v, nv).pvalue
        others = [PRED[(ds, sd, "rot")][(a, b, False)].mean() for sd in seeds5("fig5c-rotations-predict-rotations")[1:]]
        print(f"{NAME[ds]} {a} -> {b}: mean {v.mean():.3f} s.d. {v.std():.3f}; null mean {nv.mean():.3f}; signed-rank p {p:.1e} "
              f"(rank-sum {p2:.1e}); other seeds {np.round(others, 3)}")

**Result: reproduced.** Each rotation predicts on average 41.4 to 94.6% of the variance of another rotation, above the shuffled null (2.4 to 76.3%) for every pair (signed-rank p at most 4.9e-13). All 20 pair means lie in 38.7 to 100%, and the four other seeds move them by at most 0.03.

In [ ]:
for ds in DATASETS:
    for mode in ("loc", "loc_code"):
        r = PRED[(ds, s0, mode)]
        v, nv = r[("location", "rest", False)], r[("location", "rest", True)]
        extra = ""
        if mode == "loc":
            extra = "; other seeds " + str(np.round([PRED[(ds, sd, 'loc')][('location', 'rest', False)].mean() for sd in seeds5("fig5c-rotations-predict-rotations")[1:]], 3))
        print(f"{NAME[ds]} ({'Methods: 6 folds, lambda 0.1' if mode == 'loc' else 'code: 2 folds, lambda 0.01, SVD 90%'}): mean {v.mean():.3f} "
              f"s.d. {v.std():.3f}; null mean {nv.mean():.3f}; signed-rank p {stats.wilcoxon(v, nv).pvalue:.1e}{extra}")

**Result: reproduced.** The state space location predicts on average 70.9, 72.3, 80.9 and 71.6% of the population variance (6 folds, lambda 0.1), within 60.3 to 88%, against a null of 12.9 to 22.6% (signed-rank p at most 2.4e-13). The code's variant gives 73.1 to 85.6%.

## fig6c-location-tracks-direction

PCA of the offset columns across the 72 conditions; circular correlation between target angle and the
angle of each condition in the PC 1-2 plane (claims). A sign flip of PC 2 reverses the direction of
travel, so both orientations are shown.

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(14, 3.3))
for ax, ds in zip(axs, DATASETS):
    m = ds[0]
    loc = D[ds]["L"][:, :, dict(comps(ds))["location"][0]]
    Z = loc - loc.mean(0)
    U, S, Vt = np.linalg.svd(Z, full_matrices=False)
    pc = Z @ Vt[:2].T
    ang = np.arctan2(pc[:, 1], pc[:, 0])
    tang = CT[m]["angle"]
    r1, p1 = ldr.circ_corr(tang, ang)
    r2, p2 = ldr.circ_corr(tang, -ang)
    print(f"{NAME[ds]}: circular correlation {r1:.3f} p {p1:.2e} | PC 2 flipped {r2:.3f} p {p2:.2e} | PC1-2 variance share "
          f"{(S[:2] ** 2).sum() / (S ** 2).sum():.3f}")
    sc = ax.scatter(pc[:, 0], pc[:, 1], c=tang, cmap="hsv", s=14, marker="o")
    ax.set_title(NAME[ds]); ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
plt.tight_layout(); plt.show()

**Result: partially reproduced.** With PC 2 flipped, the angle of the location in the PC 1-2 plane correlates with target angle in M1-J (0.613), PMd-J (0.701) and M1-N (0.245, p = 0.016), but not in PMd-N (0.053 or -0.053, p = 0.62).

## jPCA

jPCA as in the Methods (Explaining variability in jPCA-identified rotations): activity reconstructed
from the temporal basis functions (raw SVD basis over -350 to +850 ms with the dataset's k),
soft-normalised (constant 5), mean-subtracted across conditions, 6 PCs, fitted on bins 20 to 40 of the
grid (MATLAB indices; -160 to +40 ms). Two planes are kept for M1 and one for PMd.

In [ ]:
def run_jpca(Xc_norm, conds):
    return ldr.jpca(Xc_norm[conds], np.arange(19, 40))


JP = {}
for ds in DATASETS:
    B2, L2 = ldr.tbf(D[ds]["X"], D[ds]["k"])
    Xr = np.einsum("cnk,kt->cnt", L2, B2)
    Xn, nf = ldr.soft_norm(Xr)
    Xm = Xn - Xn.mean(0)
    allc = np.arange(72)
    st = CT[ds[0]]["straight"]
    J = run_jpca(Xm, allc)
    nplanes = 2 if ds[1] == "M1" else 1
    W = [J["PC"] @ J["planes"][p] for p in range(nplanes)]
    JP[ds] = dict(J=J, W=W, Xm=Xm, nf=nf, mu_t=Xn.mean(0),
                  Jst=run_jpca(Xm, allc[st]), Jcv=run_jpca(Xm, allc[~st]))

## jpca-fit-r2

> Rotational dynamics found by jPCA fit activity within the jPCA planes with R2 of 0.64 to 0.73, higher
> on straight reaches (0.73 to 0.88) than on curved reaches (0.56 to 0.70).

In [ ]:
for ds in DATASETS:
    J, Js, Jc = JP[ds]["J"], JP[ds]["Jst"], JP[ds]["Jcv"]
    print(f"{NAME[ds]}: (a) skew R2 all {J['R2_skew_2D']:.3f}, straight {Js['R2_skew_2D']:.3f}, curved {Jc['R2_skew_2D']:.3f} | "
          f"(b) unconstrained R2 all {J['R2_best_2D']:.3f}, straight {Js['R2_best_2D']:.3f}, curved {Jc['R2_best_2D']:.3f}")
    # per-condition R2 in the first plane of the all-condition fit; straight vs curved paired by maze
    Z = (JP[ds]["Xm"][:, :, 19:40].transpose(0, 2, 1) @ J["PC"]) @ J["planes"][0]
    Ms2 = J["planes"][0].T @ J["Mskew"] @ J["planes"][0]
    r2c = []
    for c in range(72):
        pre, d2 = Z[c, :-1], np.diff(Z[c], axis=0)
        r2c.append(1 - np.sum((d2 - pre @ Ms2) ** 2) / np.sum((d2 - d2.mean(0)) ** 2))
    r2c = np.array(r2c)
    mz = CT[ds[0]]["maze"]
    si = [i for i in range(72) if CT[ds[0]]["straight"][i] and (mz[i] + 1) in mz]
    ci = [int(np.where(mz == mz[i] + 1)[0][0]) for i in si]
    print(f"   per-condition R2 (first plane): straight median {np.median(r2c[si]):.3f}, curved median {np.median(r2c[ci]):.3f}; "
          f"signed-rank over {len(si)} maze pairs p {stats.wilcoxon(r2c[si], r2c[ci]).pvalue:.2e}")

**Result: partially reproduced.** With the skew-symmetric R2 (reading a), jPCA fits 0.776, 0.664, 0.623 and 0.452 in the first plane; three lie in 0.576 to 0.803 and PMd-N does not. Refitted on straight and curved reaches alone, every dataset fits straight reaches better (0.753 to 0.843) than curved ones (0.562 to 0.712), all within their ranges. The unconstrained R2 (reading b) has two of four in range for all conditions.

## jpca-variance-explained

> Activity within the jPCA rotational planes explained only 7 to 12% of the population variance in
> peri-movement firing rates.

Per condition: the soft-normalised, mean-subtracted activity reconstructed in the retained planes,
un-normalised, its summed variance over -150 to +850 ms as a fraction of the summed variance of the
condition's activity (not mean-subtracted).

In [ ]:
for ds in DATASETS:
    W = np.column_stack(JP[ds]["W"])
    Q = ldr.orth(W)
    ve = []
    for c in range(72):
        Y = Q @ (Q.T @ JP[ds]["Xm"][c])
        Yu = Y * JP[ds]["nf"][:, None]
        X = D[ds]["X"][c]
        ve.append(np.var(Yu[:, ldr.FIT0:], axis=1).sum() / np.var(X[:, ldr.FIT0:], axis=1).sum())
    ve = np.array(ve)
    D[ds]["jpca_ve"] = ve
    print(f"{NAME[ds]} ({W.shape[1] // 2} plane(s)): mean {ve.mean():.3f} s.d. {ve.std():.3f} range {ve.min():.3f}-{ve.max():.3f}")

**Result: partially reproduced.** The retained jPCA planes hold on average 8.9, 11.6, 11.7 and 5.1% of the population variance (s.d. 3.2 to 11.6% across conditions); three lie in 6.3 to 13.2% and PMd-N (5.1%) is below.

## suppfig3-model-comparison

In [ ]:
for ds in DATASETS:
    k = D[ds]["k"]
    tbf_cv = CV[ds].mean(0)[k - 1]
    print(f"{NAME[ds]}: jPCA {D[ds]['jpca_ve'].mean():.3f} | single-condition LDS {D[ds]['lds_ve'].mean():.3f} | TBF cross-validated "
          f"(held-out VE at k={k}) {tbf_cv:.3f} | TBF in sample {D[ds]['tbf_ve'].mean():.3f}; TBF highest (cross-validated): "
          f"{tbf_cv > max(D[ds]['jpca_ve'].mean(), D[ds]['lds_ve'].mean())}; (in sample): "
          f"{D[ds]['tbf_ve'].mean() > max(D[ds]['jpca_ve'].mean(), D[ds]['lds_ve'].mean())}")

**Result: not reproduced.** Under the main reading, the cross-validated TBF value (71.1, 79.0, 86.6 and 79.1%) is above jPCA (5.1 to 11.7%) but below the in-sample single-condition LDS (84.4 to 94.2%) in every dataset. In sample (the variant), TBF (93.5 to 96.6%) exceeds both in every dataset.

## fig4d-jpca-variance-correlation and fig4f-plane-alignment-explains-jpca

The basis functions are extracted from the soft-normalised, mean-subtracted activity (-150 to +850 ms,
the dataset's k), as the Methods say. Per condition: the variance of the rotational components, the
variance in each retained jPCA plane, and the alignment index of the total rotational activity onto
the plane.

In [ ]:
J4 = {}
for ds in DATASETS:
    Xm = JP[ds]["Xm"][..., ldr.FIT0:]
    Bn, Ln = ldr.tbf(Xm, D[ds]["k"])
    Bn, Ln, lam, g, fr = ldr.demix(Bn, Ln)
    rot = np.where(g >= 0)[0]
    rv = np.array([np.var(Ln[c][:, rot] @ Bn[rot], axis=1).sum() for c in range(72)])
    J4[ds] = []
    for p, Wp in enumerate(JP[ds]["W"]):
        pv = np.array([np.var(Wp.T @ Xm[c], axis=1).sum() for c in range(72)])
        ai = np.array([ldr.alignment_index(Ln[c][:, rot], Bn[rot], Wp) for c in range(72)])
        J4[ds].append(dict(plane=p + 1, r_var=stats.pearsonr(rv, pv), r_ai=stats.pearsonr(ai, pv), npairs=int(g.max() + 1)))

In [ ]:
rhos, sig = [], []
for ds in DATASETS:
    for e in J4[ds]:
        r, p = e["r_var"]
        rhos.append(r); sig.append(p < 0.05)
        print(f"{NAME[ds]} jPCA plane {e['plane']}: rho {r:.3f} p {p:.2e} (rotation pairs in the normalised TBF {e['npairs']})")
    print(f"   {NAME[ds]} mean rho over retained planes {np.mean([e['r_var'][0] for e in J4[ds]]):.3f}")
print(f"planes with p < 0.05: {sum(sig)} of {len(sig)} ({100 * np.mean(sig):.0f}%)")

**Result: not reproduced.** Rotational variance correlates with the variance in the jPCA planes more strongly than the paper reports: rho = -0.061 and 0.711 (M1-J), 0.930 (PMd-J), 0.663 and 0.627 (M1-N), 0.682 (PMd-N). Per-dataset means are 0.325, 0.930, 0.645 and 0.682, and the significant planes have rho 0.627 to 0.930; under neither reading do the values lie in 0.351 to 0.506. Five of six planes (83%) are significant, outside 66 +- 6.6%.

In [ ]:
for ds in DATASETS:
    for e in J4[ds]:
        r, p = e["r_ai"]
        print(f"{NAME[ds]} jPCA plane {e['plane']}: rho (alignment index vs variance in plane) {r:.3f} p {p:.2e}")

**Result: partially reproduced.** The alignment of a condition's rotations with a jPCA plane correlates with its variance in that plane for every plane (p at most 2.3e-11), with rho 0.941, 0.926, 0.832, 0.689, 0.817 and 0.815; five of six lie in 0.738 to 1 (M1-N plane 1 does not).

## Kinematics

Hand position every 10 ms from -350 to +850 ms, relative to the position at -350 ms, averaged per
condition. Kinematic basis functions: the top two singular vectors of x and y hand velocity of all
conditions of both monkeys (reading (a)), integrated to position; reading (b) fits a skew-normal and the
derivative of a skew-normal to them (Methods, Extracting kinematic basis functions). Reach parameters
are the least-squares coefficients of x and y position on the two position basis functions.

In [ ]:
KIN = {}
vel = []
for m in ("J", "N"):
    H = SESS[m]["hand"][:, ::10]
    H = H - H[:, :1]
    Hc = ldr.cond_average(H, CT[m]["cond"], 72, MASK[m])
    KIN[m] = dict(H=H, Hc=Hc)
    vel.append(np.gradient(Hc, axis=1) / 0.01)
Bv, Bp_a, Sv = ldr.kin_basis(vel)
print("velocity singular values (top 5):", np.round(Sv[:5], 1), "share of the top two", np.round((Sv[:2] ** 2).sum() / (Sv ** 2).sum(), 3))
try:
    Bv_b, Bp_b = ldr.skewnorm_basis(ldr.GRID_MS.astype(float), Bv)
    print("skew-normal fit, correlation with the SVD velocity vectors:",
          [round(float(np.corrcoef(Bv[i], Bv_b[i])[0, 1]), 3) for i in range(2)])
except Exception as err:
    Bp_b = None
    print("skew-normal fit failed:", err)
KB = {"a": Bp_a, "b": Bp_b} if Bp_b is not None else {"a": Bp_a}
for m in ("J", "N"):
    for r, Bp in KB.items():
        P = ldr.reach_params(KIN[m]["Hc"], Bp)
        KIN[m]["P" + r] = P
        ve = np.mean([ldr.kin_ve(KIN[m]["Hc"][c], ldr.params_to_pos(P[c:c + 1], Bp)[0]) for c in range(72)])
        print(f"monkey {m}, reading ({r}): reach parameters reproduce condition-mean hand position with VE {ve:.3f}")
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].plot(ldr.GRID_MS, Bv.T); ax[0].set_title("velocity basis (a)")
ax[1].plot(ldr.GRID_MS, Bp_a.T); ax[1].set_title("position basis (a)")
plt.tight_layout(); plt.show()

## fig6e-kinematics-predict-location

Ridge regression (lambda 0.1, intercept) from the four reach parameters to the offset column, 6 folds
over conditions; variance explained in the offset column over held-out conditions. Kernel (a): OU,
length scale 1000, no penalty; kernel (b): Gaussian, length scale 400, lambda 0.1. Shuffle: reach
parameters permuted across conditions, 1000 permutations.

In [ ]:
def cv_offset(P, Y, folds, model):
    pred = np.zeros_like(Y)
    for te in folds:
        tr = np.setdiff1d(np.arange(len(Y)), te)
        if model == "lin":
            f = ldr.ridge_fit(P[tr], Y[tr], 0.1)
        elif model == "ou":
            f = ldr.kernel_fit(P[tr], Y[tr], "ou", 1e3, 0.0)
        else:
            f = ldr.kernel_fit(P[tr], Y[tr], "gauss", 400.0, 0.1)
        pred[te] = f(P[te])
    return 1 - np.sum((Y - pred) ** 2) / np.sum((Y - Y.mean(0)) ** 2)


def fig6e_job(args):
    ds, r, sd = args
    rng = np.random.default_rng(sd)
    P = KIN[ds[0]]["P" + r]
    Y = D[ds]["L"][:, :, dict(comps(ds))["location"][0]]
    folds = ldr.folds_of(72, 6, rng)
    out = {}
    for model in ("lin", "ou", "gauss"):
        real = cv_offset(P, Y, folds, model)
        null = np.array([cv_offset(P[rng.permutation(72)], Y, folds, model) for _ in range(1000 if sd == seed("fig6e-kinematics-predict-location") else 0)])
        out[model] = (real, null)
    return out


_jobs = [(ds, r, sd) for ds in DATASETS for r in KB for sd in seeds5("fig6e-kinematics-predict-location")]
F6E = dict(zip(_jobs, ldr.pmap(fig6e_job, _jobs)))

In [ ]:
s0 = seed("fig6e-kinematics-predict-location")
for ds in DATASETS:
    for r in KB:
        res = F6E[(ds, r, s0)]
        txt = []
        for model, lab in (("lin", "linear"), ("ou", "kernel OU"), ("gauss", "kernel Gaussian")):
            real, null = res[model]
            others = [F6E[(ds, r, sd)][model][0] for sd in seeds5("fig6e-kinematics-predict-location")[1:]]
            txt.append(f"{lab} {real:.3f} (null max {null.max():.3f}, p {(1 + (null >= real).sum()) / (1 + len(null)):.4f}; "
                       f"other seeds {np.round(others, 3)})")
        print(f"{NAME[ds]} kinematic reading ({r}): " + " | ".join(txt))

**Result: reproduced.** With the SVD kinematic basis (reading a), the four reach parameters linearly predict 53.3, 55.0, 70.4 and 41.4% of the variance of the location (within 33.3 to 78.1%), and OU kernel regression 70.4, 68.8, 88.7 and 76.9% (within 62.1 to 89.1%), both above all 1000 permutations. The skew-normal basis (reading b) gives the same within 0.02. The Gaussian kernel (reading b of the kernel) gives 32 to 54%, below the range.

## suppfig2-eigenvalues-from-kinematics

> Kinematic parameters predicted the eigenvalues poorly (mean R2 = 0.1, leave-one-out cross-validation).

Predictors: the four reach parameters, the condition's mean reach duration and a constant; targets:
real and imaginary parts of each frequency-sorted eigenvalue of the single-condition LDSs; OLS, leave
one condition out; R2 per component with negatives set to 0; mean over components pooled across the
four datasets. Components with no variance (the imaginary part of a real eigenvalue) are left out.

In [ ]:
for r in KB:
    allr2 = []
    for ds in DATASETS:
        lams = [ldr.sort_eigs(l) for l in D[ds]["lds_lam"]]
        n = min(len(l) for l in lams)
        Y = np.column_stack([np.array([l[:n] for l in lams]).real, np.array([l[:n] for l in lams]).imag])
        Y = Y[:, Y.std(0) > 1e-12]
        Xp = np.column_stack([KIN[ds[0]]["P" + r], D[ds]["cdur"], np.ones(72)])
        pred = np.zeros_like(Y)
        for c in range(72):
            o = np.arange(72) != c
            w = np.linalg.lstsq(Xp[o], Y[o], rcond=None)[0]
            pred[c] = Xp[c] @ w
        r2 = np.maximum(0, 1 - ((Y - pred) ** 2).sum(0) / ((Y - Y.mean(0)) ** 2).sum(0))
        allr2.append(r2)
        print(f"reading ({r}) {NAME[ds]}: {len(r2)} components, mean R2 {r2.mean():.3f}, max {r2.max():.3f}")
    print(f"reading ({r}): mean R2 pooled over datasets {np.concatenate(allr2).mean():.3f}")

**Result: not reproduced.** Reach parameters and reach duration predict the eigenvalue components with a mean leave-one-out R2 of 0.275 pooled over datasets (0.135 to 0.480 per dataset) under both kinematic readings, above 0.1 +- 0.05.

## fig7c-ldr-encoding and suppfig7a-not-memorization

LDR encoding (Methods, LDR encoding; claims): 6 folds over conditions with TBF and loadings refitted on
5/6; ridge (lambda 0.1, intercept) or OU kernel regression (length scale 1000, no penalty) from the
reach parameters to the vectorised loading matrix; predicted loadings times the training basis;
population variance explained per held-out condition. Null: the model trained with reach parameters
permuted across the training conditions, signed-rank test over conditions.

In [ ]:
def encode(ds, P, folds, model, rng=None, null=False):
    ve = np.zeros(72)
    for te in folds:
        tr = np.setdiff1d(np.arange(72), te)
        B, L = ldr.tbf(D[ds]["Xw"][tr], D[ds]["k"])
        Y = L.reshape(len(tr), -1)
        Ptr = P[tr][rng.permutation(len(tr))] if null else P[tr]
        if model == "lin":
            f = ldr.ridge_fit(Ptr, Y, 0.1)
        elif model == "ou":
            f = ldr.kernel_fit(Ptr, Y, "ou", 1e3, 0.0)
        else:  # nearest-neighbour regression
            f = lambda Z, Ptr=Ptr, Y=Y: Y[np.argmin(((Z[:, None] - Ptr[None]) ** 2).sum(2), 1)]
        Lh = f(P[te]).reshape(len(te), -1, B.shape[0])
        for q, c in enumerate(te):
            ve[c] = ldr.pop_ve(D[ds]["Xw"][c], Lh[q] @ B)
    return ve


def enc_job(args):
    ds, r, sd = args
    rng = np.random.default_rng(sd)
    P = KIN[ds[0]]["P" + r]
    folds = ldr.folds_of(72, 6, rng)
    out = {m: encode(ds, P, folds, m) for m in ("lin", "ou")}
    out.update({m + "_null": encode(ds, P, folds, m, rng, True) for m in ("lin", "ou")})
    ang = CT[ds[0]]["angle"]
    sext = np.floor(((ang + np.pi) % (2 * np.pi)) / (np.pi / 3)).astype(int)
    sfolds = [np.where(sext == s)[0] for s in range(6) if (sext == s).any()]
    out["sext_ou"] = encode(ds, P, sfolds, "ou")
    out["sext_nn"] = encode(ds, P, sfolds, "nn")
    return out


_jobs = [(ds, r, sd) for ds in DATASETS for r in KB for sd in seeds5("fig7c-ldr-encoding")]
ENC = dict(zip(_jobs, ldr.pmap(enc_job, _jobs)))

In [ ]:
s0 = seed("fig7c-ldr-encoding")
for r in KB:
    for ds in DATASETS:
        e = ENC[(ds, r, s0)]
        o = {m: [ENC[(ds, r, sd)][m].mean() for sd in seeds5("fig7c-ldr-encoding")[1:]] for m in ("lin", "ou")}
        print(f"reading ({r}) {NAME[ds]}: linear mean {e['lin'].mean():.3f} s.d. {e['lin'].std():.3f} (null {e['lin_null'].mean():.3f}, "
              f"signed-rank p {stats.wilcoxon(e['lin'], e['lin_null']).pvalue:.1e}; other seeds {np.round(o['lin'], 3)}) | "
              f"kernel mean {e['ou'].mean():.3f} s.d. {e['ou'].std():.3f} (null {e['ou_null'].mean():.3f}, p "
              f"{stats.wilcoxon(e['ou'], e['ou_null']).pvalue:.1e}; other seeds {np.round(o['ou'], 3)})")

**Result: reproduced.** Linear LDR encoding explains on average 76.5, 68.1, 84.3 and 72.3% of the population variance (within 54 to 85.8%) and kernel LDR encoding 84.0, 85.0, 92.1 and 86.7% (within 63.9 to 95.7%), both above the shuffled null (p at most 2.6e-08). The skew-normal basis gives the same within 0.01.

In [ ]:
for r in KB:
    for ds in DATASETS:
        e = ENC[(ds, r, s0)]
        print(f"reading ({r}) {NAME[ds]}: sextant hold-out, kernel mean {e['sext_ou'].mean():.3f}, nearest neighbour mean "
              f"{e['sext_nn'].mean():.3f}; kernel higher on {int((e['sext_ou'] > e['sext_nn']).sum())}/72; signed-rank p "
              f"{stats.wilcoxon(e['sext_ou'], e['sext_nn']).pvalue:.1e}")

**Result: partially reproduced.** With target sextants held out, kernel regression beats nearest-neighbour regression with p < 0.01 in M1-J, M1-N and PMd-N (p 2e-11 to 6e-10), but not in PMd-J (p = 0.019 under the SVD basis, 0.89 under the skew-normal basis).

## fig7c-standard-encoding-worse

Standard instantaneous encoding models (Methods, Comparing the LDR encoding with standard instantaneous
encoding models), 100 ms causal lag, the same 6 folds of conditions as fig7c: (1) ridge regression
(lambda 0.1) on position, extent, velocity, speed, velocity angle (as its cosine and sine; assumption)
and acceleration, with a baseline; (2) OU kernel regression on position, length scale 1e12; (3) OU
kernel regression on velocity, length scale 1e11. The kernel penalty is not given; 0.1 is used
(assumption, the Methods' L2 for kernel decoders).

In [ ]:
def kin_features(m):
    Hc = KIN[m]["Hc"]
    v = np.gradient(Hc, axis=1) / 0.01
    a = np.gradient(v, axis=1) / 0.01
    ext = np.linalg.norm(Hc, axis=2)[..., None]
    sp = np.linalg.norm(v, axis=2)[..., None]
    ang = np.arctan2(v[..., 1], v[..., 0])
    lin = np.concatenate([Hc, ext, v, sp, np.cos(ang)[..., None], np.sin(ang)[..., None], a], axis=2)
    return dict(lin=lin, pos=Hc, vel=v)


def std_job(args):
    ds, sd = args
    rng = np.random.default_rng(sd)
    folds = ldr.folds_of(72, 6, rng)
    feats = kin_features(ds[0])
    lag = 10  # 100 ms at 10 ms bins
    tt = np.arange(ldr.FIT0, len(ldr.GRID_MS))
    out = {}
    for name in ("lin", "pos", "vel"):
        F = feats[name][:, tt - lag]  # (c, T, f)
        mu, sdv = F.reshape(-1, F.shape[2]).mean(0), F.reshape(-1, F.shape[2]).std(0) + 1e-12
        ve = np.zeros(72)
        for te in folds:
            tr = np.setdiff1d(np.arange(72), te)
            Xtr = F[tr].reshape(-1, F.shape[2])
            Ytr = D[ds]["Xw"][tr].transpose(0, 2, 1).reshape(-1, D[ds]["Xw"].shape[1])
            if name == "lin":
                f = ldr.ridge_fit((Xtr - mu) / sdv, Ytr, 0.1)
                pr = lambda Z: f((Z - mu) / sdv)
            else:
                f = ldr.kernel_fit(Xtr, Ytr, "ou", 1e12 if name == "pos" else 1e11, 0.1)
                pr = f
            for c in te:
                ve[c] = ldr.pop_ve(D[ds]["Xw"][c], pr(F[c]).T)
        out[name] = ve
    return out


STD = dict(zip(DATASETS, ldr.pmap(std_job, [(ds, seed("fig7c-ldr-encoding")) for ds in DATASETS])))

In [ ]:
for r in KB:
    nwin = 0
    for ds in DATASETS:
        e = ENC[(ds, r, s0)]
        for name, lab in (("lin", "linear multi-parameter"), ("pos", "kernel on position"), ("vel", "kernel on velocity")):
            sv = STD[ds][name]
            res = []
            for ldrm in ("lin", "ou"):
                p = stats.wilcoxon(e[ldrm], sv).pvalue
                win = (e[ldrm].mean() > sv.mean()) and p < 0.001
                nwin += win
                res.append(f"LDR {ldrm} {e[ldrm].mean():.3f} p {p:.1e}")
            print(f"reading ({r}) {NAME[ds]} {lab}: mean {sv.mean():.3f} | " + "; ".join(res))
    print(f"reading ({r}): LDR higher with p < 0.001 in {nwin} of 24 comparisons")

**Result: reproduced.** Both LDR models exceed each standard encoding model in every dataset with p < 0.001 (24 of 24 comparisons under both kinematic readings). The linear multi-parameter model explains 51.7 to 53.4%. The two OU kernel models explain 0% with the assumed penalty of 0.1: with length scales of 1e11 and 1e12 the kernel is nearly constant and they predict the mean, so the comparison with them is not informative.

## fig7f-ldr-decoding, trial-curvature-decoding, suppfig8-no-output-potent-dependence

Two folds over conditions. On the training half: TBF on trial averages (raw SVD basis, -150 to +850
ms) and a ridge map (lambda 10, intercept) from vectorised loadings to the four reach parameters. On
the held-out half, single-trial loadings are 10 ms spike counts in spikes/s times the pseudo-inverse of
the basis; predicted parameters times the position basis give hand position, compared with the trial's
own hand position (relative to -350 ms) over -350 to +850 ms (claims; code
kinFromTBFSingleTrial.m). Variance explained per trial, negatives set to 0.

In [ ]:
def decode(ds, r, sd, lam=10.0, remove_potent=False, null=False):
    m = ds[0]
    rng = np.random.default_rng(sd)
    Bp = KB[r]
    P = KIN[m]["P" + r]
    folds = ldr.folds_of(72, 2, rng)
    cond = D[ds]["cond"]
    C = SESS[m]["C10"][:, D[ds]["u"]][..., ldr.FIT0:]
    Xw = D[ds]["Xw"]
    if remove_potent:
        # output-potent dimensions: row space of least-squares maps from activity to hand position and velocity
        Hc = KIN[m]["Hc"][:, ldr.FIT0:]
        Vc = (np.gradient(KIN[m]["Hc"], axis=1) / 0.01)[:, ldr.FIT0:]
        Xs = Xw.transpose(0, 2, 1).reshape(-1, Xw.shape[1])
        K = np.concatenate([Hc, Vc], axis=2).reshape(-1, 4)
        Wm = np.linalg.lstsq(Xs - Xs.mean(0), K - K.mean(0), rcond=None)[0]
        Q = ldr.orth(Wm)
        Xw = Xw - np.einsum("nk,mk,cmt->cnt", Q, Q, Xw)
        C = C - np.einsum("nk,mk,cmt->cnt", Q, Q, C)
    ve = np.full(len(cond), np.nan)
    dec = np.full((len(cond), 4), np.nan)
    for te in folds:
        tr = np.setdiff1d(np.arange(72), te)
        B, L = ldr.tbf(Xw[tr], D[ds]["k"])
        Ptr = P[tr][rng.permutation(len(tr))] if null else P[tr]
        f = ldr.ridge_fit(L.reshape(len(tr), -1), Ptr, lam)
        Bpinv = np.linalg.pinv(B)
        ii = np.where(MASK[m] & np.isin(cond, te))[0]
        Lt = C[ii] @ Bpinv
        ph = f(Lt.reshape(len(ii), -1))
        pos = ldr.params_to_pos(ph, Bp)
        dec[ii] = ph
        for q, i in enumerate(ii):
            ve[i] = ldr.kin_ve(KIN[m]["H"][i], pos[q])
    return ve, dec


def dec_job(args):
    return decode(*args[:3], **args[3])


_jobs = [(ds, r, sd, {}) for ds in DATASETS for r in KB for sd in seeds5("fig7f-ldr-decoding")]
_jobs += [(ds, r, seed("fig7f-ldr-decoding"), {"null": True}) for ds in DATASETS for r in KB]
_jobs += [(ds, r, seed("suppfig8-no-output-potent-dependence"), {"lam": 1.0, "remove_potent": rp}) for ds in DATASETS for r in KB for rp in (False, True)]
DEC = {(a[0], a[1], a[2], tuple(sorted(a[3].items()))): v for a, v in zip(_jobs, ldr.pmap(dec_job, _jobs))}

In [ ]:
s0 = seed("fig7f-ldr-decoding")
for r in KB:
    for ds in DATASETS:
        ve, _ = DEC[(ds, r, s0, ())]
        nv, _ = DEC[(ds, r, s0, (("null", True),))]
        ok = np.isfinite(ve)
        others = [np.nanmean(DEC[(ds, r, sd, ())][0]) for sd in seeds5("fig7f-ldr-decoding")[1:]]
        print(f"reading ({r}) {NAME[ds]}: {int(ok.sum())} trials, mean VE {np.nanmean(ve):.3f} s.d. {np.nanstd(ve):.3f} median "
              f"{np.nanmedian(ve):.3f}; condition-shuffled null mean {np.nanmean(nv):.3f} (signed-rank p "
              f"{stats.wilcoxon(ve[ok], nv[ok]).pvalue:.1e}); other seeds {np.round(others, 3)}")

**Result: partially reproduced.** The LDR decoder explains on average 77.4, 74.7, 87.4 and 73.4% of the variance of single-trial hand position (SVD basis; medians 82 to 92%); M1-J and M1-N lie in 75.6 to 100%, PMd-J and PMd-N below. The skew-normal basis gives 72.9 to 86.8%, with the same two in range. Under the four other seeds PMd-N reaches 75.8 and 76.1% twice, so two or three of four lie in range under every seed.

In [ ]:
for r in KB:
    for ds in DATASETS:
        m = ds[0]
        _, dec = DEC[(ds, r, s0, ())]
        ok = np.isfinite(dec[:, 0])
        cond = D[ds]["cond"]
        true = ldr.reach_params(KIN[m]["H"], KB[r])
        out = []
        for j in (2, 3):
            dres = dec[ok, j] - np.array([np.mean(dec[ok & (cond == c), j]) for c in cond[ok]])
            tres = true[ok, j] - np.array([np.mean(true[ok & (cond == c), j]) for c in cond[ok]])
            rr, pp = stats.pearsonr(dres, tres)
            out.append(f"parameter {j + 1}: rho {rr:.3f} p {pp:.1e}")
        print(f"reading ({r}) {NAME[ds]} ({int(ok.sum())} trials): " + "; ".join(out))

**Result: partially reproduced.** Decoded and actual within-condition curvature correlate with p < 0.001 for all eight parameters, with rho 0.319, 0.162, 0.229, 0.093, 0.449, 0.548, 0.191 and 0.290 (SVD basis); 6 of 8 lie in 0.144 to 0.451 (PMd-J parameter 4 is lower, M1-N parameter 4 higher).

In [ ]:
s8 = seed("suppfig8-no-output-potent-dependence")
for r in KB:
    for ds in DATASETS:
        cond = D[ds]["cond"]
        a, _ = DEC[(ds, r, s8, (("lam", 1.0), ("remove_potent", False)))]
        b, _ = DEC[(ds, r, s8, (("lam", 1.0), ("remove_potent", True)))]
        ca = np.array([np.nanmean(a[cond == c]) for c in range(72)])
        cb = np.array([np.nanmean(b[cond == c]) for c in range(72)])
        p = stats.wilcoxon(ca, cb).pvalue
        print(f"reading ({r}) {NAME[ds]}: per-condition VE with potent dims {ca.mean():.3f}, without {cb.mean():.3f} "
              f"(mean drop {np.mean(ca - cb):.3f}); two-sided signed-rank p {p:.3f}")

**Result: not reproduced.** Removing the dimensions that map to hand position and velocity (the dataset has no EMG) lowers the decoded variance by 3.9 to 6.3 points per condition, and the two-sided signed-rank test finds the drop in every dataset (p < 0.001, below 0.036).

## fig7f-decoder-beats-standard

Instantaneous decoders of hand position (Methods, Comparing LDR-based decoding to instantaneous
decoding methods): activity 100 ms before each time point, smoothed with a 20 ms Gaussian (the
single-trial rates used throughout) or a 100 ms Gaussian (10 ms counts smoothed with s.d. 10 bins, a
standard approximation); ridge regression (lambda 1) trained on single trials; OU kernel regression
(length scale 100, L2 0.1) trained on a random subset of 3000 single-trial time points (the Methods'
"subset of single-trial data"; the size is an assumption) or on trial averages. The same two folds of
conditions as the LDR decoder; variance explained in position per trial over -250 to +850 ms, compared
with the LDR decoder's per-trial values by two-sided signed-rank tests. The GPFA decoders are not run:
the paper gives neither the latent dimensionality nor the bin size, and there was not time to fit them.

In [ ]:
from scipy.ndimage import gaussian_filter1d


def inst_job(args):
    ds, r, sd = args
    m = ds[0]
    rng = np.random.default_rng(sd)
    folds = ldr.folds_of(72, 2, rng)  # the same draw as decode() under this seed
    rng2 = np.random.default_rng(seed("fig7f-decoder-beats-standard"))
    cond = D[ds]["cond"]
    u = D[ds]["u"]
    lag = 10
    tt = np.arange(lag, len(ldr.GRID_MS))
    feats = {"20 ms": SESS[m]["R"][:, u], "100 ms": gaussian_filter1d(SESS[m]["C10"][:, u], 10, axis=2, mode="reflect")}
    H = KIN[m]["H"]
    out = {}
    for sname, F in feats.items():
        for dname in ("linear, single trials", "kernel, single trials", "kernel, trial averages"):
            ve = np.full(len(cond), np.nan)
            for te in folds:
                tr = np.setdiff1d(np.arange(72), te)
                itr = np.where(MASK[m] & np.isin(cond, tr))[0]
                ite = np.where(MASK[m] & np.isin(cond, te))[0]
                if dname == "kernel, trial averages":
                    Fa = ldr.cond_average(F, cond, 72, MASK[m])[tr]
                    Ha = KIN[m]["Hc"][tr]
                    X = Fa[:, :, tt - lag].transpose(0, 2, 1).reshape(-1, F.shape[1])
                    Y = Ha[:, tt].reshape(-1, 2)
                else:
                    X = F[itr][:, :, tt - lag].transpose(0, 2, 1).reshape(-1, F.shape[1])
                    Y = H[itr][:, tt].reshape(-1, 2)
                mu, sdv = X.mean(0), X.std(0) + 1e-9
                X = (X - mu) / sdv
                if dname.startswith("linear"):
                    f = ldr.ridge_fit(X, Y, 1.0)
                else:
                    if len(X) > 3000:
                        pick = rng2.choice(len(X), 3000, replace=False)
                        X, Y = X[pick], Y[pick]
                    f = ldr.kernel_fit(X, Y, "ou", 1e2, 0.1)
                for i in ite:
                    Z = (F[i][:, tt - lag].T - mu) / sdv
                    ve[i] = ldr.kin_ve(H[i][tt], f(Z))
            out[(sname, dname)] = ve
    return out


_jobs = [(ds, "a", seed("fig7f-ldr-decoding")) for ds in DATASETS]
INST = dict(zip(DATASETS, ldr.pmap(inst_job, _jobs)))

In [ ]:
nwin, ntot = 0, 0
for ds in DATASETS:
    ldr_ve, _ = DEC[(ds, "a", seed("fig7f-ldr-decoding"), ())]
    for key, ve in INST[ds].items():
        ok = np.isfinite(ve) & np.isfinite(ldr_ve)
        p = stats.wilcoxon(ldr_ve[ok], ve[ok]).pvalue
        win = (np.mean(ldr_ve[ok] - ve[ok]) > 0) and p < 0.001
        nwin += win
        ntot += 1
        print(f"{NAME[ds]} {key[1]}, {key[0]} smoothing: mean {np.mean(ve[ok]):.3f} against LDR {np.mean(ldr_ve[ok]):.3f} "
              f"({int(ok.sum())} trials); two-sided signed-rank p {p:.1e}")
print(f"LDR higher with p < 0.001 in {nwin} of {ntot} comparisons (GPFA decoders not run)")

**Result: partially reproduced.** The LDR decoder explains more of the single-trial hand position than each of the 24 instantaneous decoders run (ridge regression and OU kernel regression on 20 or 100 ms smoothed activity, trained on single trials or trial averages), with two-sided signed-rank p below 1e-180 in every comparison; the best of them reaches 73.1% (M1-N, kernel on single trials, 100 ms) against 87.4% for LDR. The GPFA decoders were not run, so the claim holds against all but the GPFA decoders.

## Summary

| Claim | Outcome |
|---|---|
| data-composition | partially reproduced (units are one per electrode, unlabelled) |
| jpca-fit-r2 | partially reproduced (3 of 4 in range) |
| jpca-variance-explained | partially reproduced (3 of 4 in range) |
| single-lds-variance | not reproduced (77 to 82%, above 41 to 73%) |
| fig2c-condition-lds-variance | reproduced |
| fig3a-eigenvalue-clusters | partially reproduced (clusters present; frequencies match in PMd-J only) |
| fig3c-eigenvalue-distance-auc | not reproduced (AUC 0.63 to 0.81) |
| suppfig2-eigenvalues-from-kinematics | not reproduced (mean R2 0.275) |
| fig3g-basis-frequencies | partially reproduced (rotations near 0.7, 1.6 to 1.8 and 2.7 to 3.2 Hz) |
| tbf-variance-explained | reproduced |
| tbf-shuffle-control | partially reproduced (3 of 4 in range) |
| fig3h-recoverability | partially reproduced (chance level higher) |
| fig3i-duration-warping | not reproduced |
| suppfig3-model-comparison | not reproduced under the main reading; holds in sample |
| fig4b-plane-alignment | reproduced |
| fig4c-subspace-excursion-angles | partially reproduced |
| rotation-orientation-dimensionality | not reproduced (lower dimensional) |
| fig4d-jpca-variance-correlation | not reproduced (stronger correlations) |
| fig4f-plane-alignment-explains-jpca | partially reproduced (5 of 6 planes) |
| fig5a-frequencies-segregated | partially reproduced (18 of 20 comparisons) |
| fig5c-rotations-predict-rotations | reproduced |
| suppfig6a-location-alignment | partially reproduced (3 of 4) |
| suppfig6b-location-dimensionality | not reproduced |
| suppfig6c-location-distinct-from-rotations | reproduced |
| fig6b-location-predicts-rotations | reproduced |
| fig6c-location-tracks-direction | partially reproduced (3 of 4) |
| fig6e-kinematics-predict-location | reproduced |
| fig7c-ldr-encoding | reproduced |
| suppfig7a-not-memorization | partially reproduced (3 of 4) |
| fig7c-standard-encoding-worse | reproduced (kernel standard models uninformative) |
| fig7f-ldr-decoding | partially reproduced (2 of 4) |
| fig7f-decoder-beats-standard | partially reproduced (GPFA decoders not run) |
| suppfig8-no-output-potent-dependence | not reproduced (without EMG) |
| trial-curvature-decoding | partially reproduced (6 of 8) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

**How the analysis differs from the paper's.** The dandiset's units are one per electrode (96 per
array) with no single- or multi-unit label, where the paper used 79 to 118 sorted single and stable
multi-units per array; every population measure here rests on these units. Trials are the successful
trials of the 72 conditions with a delay above 400 ms, the reading closest to the paper's counts (an
assumption; the paper gives no trial selection). The eigenvalue nulls use 100 repeats, not 500, and the
Poisson null is simulated at 10 ms. The paper's code is not run (it needs preprocessed files that are
not public); the methods follow the claims file, the Methods, and the code as the claims file reports
it. The dataset has no EMG, so suppfig8 removes the kinematic dimensions only. The GPFA decoders
were not run.

**What was found about the dataset.** The Nitschke file joins five recording segments whose clocks
overlap: trial start times, hand timestamps and every unit's spike times jump backwards four times, so
spike times are not sorted and a trial's spikes cannot be found by time alone. Assigning each trial to
its segment in file order gives condition decoding well above chance in every segment and at chance
with shifted labels, so the alignment within segments is consistent with the trial table; condition
information in segment 3 is lower within the segment (0.16) than in segments 0 to 2 (0.41 to 0.46),
while a decoder trained on the other segments still reaches 0.48 there. In the Jenkins file one unit's
spike train restarts at the beginning of the session (two runs concatenated). Hand position is stored as
"meters" with a conversion of NaN, but its values are in millimetres (reaches of about 130). The units
table has no quality, sorting or single/multi-unit column.